In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from Bio import SeqIO

from gtf_utils import gtf_to_df_with_genes
from peptide_utils import (
    mapping_file_to_df,
    get_gene_protein_specific_peps,
    check_peptide_loc,
    count_expected_peptides_with_missed_cleavages,
    calculate_sequence_coverage
)
from plotting import (
    plot_protein_metrics
)

In [2]:
gtf_file='/home/students/q.abbas/GAPMS/0-raw_files/GCF_000499845.2_Phaseolus_vulgaris/inputs/galba/GCF_000499845.2_Phaseolus_vulgaris.galba.gtf'
prediction_fasta='/home/students/q.abbas/GAPMS/0-raw_files/GCF_000499845.2_Phaseolus_vulgaris/inputs/galba/GCF_000499845.2_Phaseolus_vulgaris.galba.faa'
mapping_file='/home/students/q.abbas/GAPMS/0-raw_files/GCF_000499845.2_Phaseolus_vulgaris/inputs/galba/GCF_000499845.2_Phaseolus_vulgaris.galba.proteomapper_out.tsv'
psauron_csv='/home/students/q.abbas/GAPMS/0-raw_files/GCF_000499845.2_Phaseolus_vulgaris/inputs/galba/GCF_000499845.2_Phaseolus_vulgaris.galba.faa.psauron.csv'
output_dir='/home/students/q.abbas/GAPMS/0-raw_files/GCF_000499845.2_Phaseolus_vulgaris/outputs/gapms/galba/'
esm_csv='/home/students/q.abbas/GAPMS/0-raw_files/GCF_000499845.2_Phaseolus_vulgaris/inputs/galba/GCF_000499845.2_Phaseolus_vulgaris.galba.faa.esm2_scores.csv'

In [3]:
gtf_file = Path(gtf_file)
protein_fasta = Path(prediction_fasta)
mapping_file = Path(mapping_file)
psauron_csv = Path(psauron_csv)
output_dir=Path(output_dir)
esm_csv = Path(esm_csv) if esm_csv else None

gtf_df = gtf_to_df_with_genes(gtf_file)
transcripts_df = gtf_df[gtf_df['Type'].isin(['transcript', 'mRNA'])][['Protein', 'Prediction_score']]
transcripts_df = transcripts_df.rename(columns={"Prediction_score": "transcript_score"})

In [4]:
# Filter for CDS features
cds_df = gtf_df.loc[gtf_df['Type'] == 'CDS'].copy()

# Compute isoforms per gene
isoforms = (
    cds_df.groupby('Gene')['Protein']
    .nunique()
    .reset_index(name='Isoforms')
)

# Compute protein start and end positions
protein_coords = (
    cds_df.groupby('Protein')
    .agg(Protein_Start=('Start', 'min'),
         Protein_End=('End', 'max'))
    .reset_index()
)

# Merge precomputed info
cds_df = cds_df.merge(protein_coords, on='Protein').merge(isoforms, on='Gene')
cds_df = cds_df.merge(transcripts_df, on='Protein', how='left')

# Prepare CDS dataframe for codon positions
df_CDSs = (
    cds_df.sort_values(['Protein', 'Start'])
          .drop_duplicates(subset=['Seqid', 'Start', 'End', 'Protein'])
          .copy()
)
df_CDSs['cds_start'] = 0
df_CDSs['cds_end'] = 0

# Calculate CDS lengths in codons
df_CDSs['cds_len'] = ((df_CDSs['End'] - df_CDSs['Start']) / 3).round().astype(int)

# Forward strand (+)
forward = df_CDSs['Strand'] == '+'
df_CDSs.loc[forward, 'cds_start'] = (
    df_CDSs[forward].groupby('Protein')['cds_len'].cumsum() - df_CDSs[forward]['cds_len'] + 1
)
df_CDSs.loc[forward, 'cds_end'] = df_CDSs[forward].groupby('Protein')['cds_len'].cumsum()

# Reverse strand (-)
reverse = ~forward
reverse_group = df_CDSs[reverse].groupby('Protein')['cds_len']
cumsum_reverse = reverse_group.cumsum()
max_len = reverse_group.transform('sum')
df_CDSs.loc[reverse, 'cds_end'] = max_len - cumsum_reverse + df_CDSs.loc[reverse, 'cds_len']
df_CDSs.loc[reverse, 'cds_start'] = df_CDSs.loc[reverse, 'cds_end'] - df_CDSs.loc[reverse, 'cds_len'] + 1

# Compute number of splice sites per protein
df_CDSs['splice_sites'] = df_CDSs.groupby('Protein')['Protein'].transform('count') - 1

# Merge codon positions back into original CDS dataframe
cds_df = cds_df.merge(
    df_CDSs[['Protein', 'Start', 'cds_start', 'cds_end', 'splice_sites']],
    on=['Protein', 'Start'],
    how='left'
)

# Ensure correct datatype
cds_df = cds_df.astype({'cds_start': 'Int64', 'cds_end': 'Int64'})

In [5]:
# 1. Load mapping
mapping_df = mapping_file_to_df(mapping_file)

# 2. Get gene-protein specific peptides
pep_df = get_gene_protein_specific_peps(cds_df, mapping_df)

# 3. Create protein sequence dictionary
seq_dict = {record.id: str(record.seq).replace('I', 'L').replace('*', '') 
            for record in SeqIO.parse(prediction_fasta, "fasta")}

# 4. Map protein sequences
pep_df['Prot_seq'] = pep_df['Protein'].map(seq_dict)
pep_df.dropna(subset=['Prot_seq'], inplace=True)

# 5. Compute peptide positions (vectorized using list comprehension)
pep_starts = []
pep_ends = []

for prot_seq, peptide in zip(pep_df['Prot_seq'], pep_df['Peptide']):
    start = prot_seq.find(peptide) + 1  # 1-based
    end = start + len(peptide) - 1 if start > 0 else 0
    pep_starts.append(start)
    pep_ends.append(end)

pep_df['pep_start'] = pep_starts
pep_df['pep_end'] = pep_ends
pep_df['prot_len'] = pep_df['Prot_seq'].str.len()

# 6. Drop temporary column
pep_df.drop(columns='Prot_seq', inplace=True)

# 7. Compute sequence coverage
coverage_series = pep_df.groupby("Protein").apply(calculate_sequence_coverage)
pep_df['sequence_coverage'] = pep_df['Protein'].map(coverage_series)


Total peptides loaded: 128940


/tmp/ipykernel_1450091/3306450161.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  coverage_series = pep_df.groupby("Protein").apply(calculate_sequence_coverage)


In [8]:
peptide_features_df = pd.merge(cds_df, pep_df, on=['Protein', 'Gene'])
peptide_features_df = peptide_features_df[[
    'Peptide', 'Protein', 'Gene', 'Isoforms', 'splice_sites', 'Protein_specific',
    'Gene_specific', 'cds_start', 'cds_end', 'peptide_length',
    'pep_start', 'pep_end', 'transcript_score', 'prot_len', 'sequence_coverage'
]]

cds_dict = peptide_features_df.groupby('Protein')[['cds_start', 'cds_end']].apply(lambda x: list(x.itertuples(index=False, name=None))).to_dict()
peptide_features_df['Splice_peptide'] = peptide_features_df.apply(lambda row: check_peptide_loc(row, cds_dict), axis=1)

peptide_features_df['N_terminal_peptide'] = np.where(peptide_features_df['pep_start'] == 1, '+', '-')
peptide_features_df['C_terminal_peptide'] = np.where(peptide_features_df['pep_end'] == peptide_features_df['prot_len'], '+', '-')
peptide_features_df = peptide_features_df.drop(columns=['cds_start', 'cds_end']).drop_duplicates()

In [ ]:
proteins_scores_df = peptide_features_df.groupby("Protein").agg(
    protein_length=("prot_len", "mean"),
    Isoforms=("Isoforms", "mean"),
    splice_sites=("splice_sites", "mean"),
    transcript_score=("transcript_score", "mean"),
    sequence_coverage=("sequence_coverage", "mean"),
    mapped_peptides=("Peptide", "count"),
    N_terminal_peptides=("N_terminal_peptide", lambda x: (x == "+").sum()),
    C_terminal_peptides=("C_terminal_peptide", lambda x: (x == "+").sum()),
    protein_specific_peptides=("Protein_specific", lambda x: (x == "+").sum()),
    gene_specific_peptides=("Gene_specific", lambda x: (x == "+").sum()),
    splice_peptides=("Splice_peptide", lambda x: (x == "+").sum()),
    internal_peptides=("Splice_peptide", lambda x: (x == "-").sum())
).reset_index()

In [11]:
other_proteins = cds_df[~cds_df['Protein'].isin(proteins_scores_df['Protein'])]
other_proteins = other_proteins[['Protein', 'transcript_score', 'Isoforms', 'splice_sites']].drop_duplicates().copy()


other_proteins['Prot_seq'] = other_proteins['Protein'].map(seq_dict)
other_proteins.dropna(subset=['Prot_seq'], inplace=True)
other_proteins['protein_length'] = other_proteins['Prot_seq'].str.len()
other_proteins.drop(columns='Prot_seq', inplace=True)

for col in proteins_scores_df.columns.difference(['Protein', 'protein_length', 'Isoforms', 'splice_sites', 'transcript_score']):
    other_proteins[col] = 0
all_proteins_scores_df = pd.concat([proteins_scores_df, other_proteins], ignore_index=True)

psauron_df = pd.read_csv(psauron_csv, sep=',', names=['Protein', 'is_protein', 'psauron_score'], on_bad_lines='skip', skip_blank_lines=True).drop(columns='is_protein')
psauron_df['psauron_score'] = pd.to_numeric(psauron_df['psauron_score'], errors='coerce')
all_proteins_scores_df = pd.merge(all_proteins_scores_df, psauron_df, on='Protein', how='left').fillna(0)

if esm_csv:
    esm_df = pd.read_csv(esm_csv, sep=',', names=['Protein','is_protein','esm_score'], on_bad_lines='skip', skip_blank_lines=True).drop(columns='is_protein')
else:
    esm_df = pd.DataFrame(columns=['Protein', 'esm_score'])
    esm_df['Protein'] = all_proteins_scores_df['Protein']
    esm_df['esm_score'] = 0
esm_df['esm_score'] = pd.to_numeric(esm_df['esm_score'], errors='coerce')
all_proteins_scores_df = pd.merge(all_proteins_scores_df, esm_df, on='Protein', how='left').fillna(0)

all_proteins_scores_df.sort_values(by='mapped_peptides', ascending=False, inplace=True)
all_proteins_scores_df.to_csv(output_dir / 'all_proteins_scores.tsv', sep='\t', index=False)